# Data Quality Check Pipeline (GX + ETL Config)
This Jupyter notebook performs data quality validation checks across ETL stages using configuration files in `config/` and `gx/`.

In [ ]:
import os
import pandas as pd
import great_expectations as gx
from utils.dq_helper import DQPipelineManager

# Initialize Data Quality Pipeline Manager
dq_manager = DQPipelineManager()
print(f"Pipeline Loaded: {dq_manager.config['pipeline']['name']}")
print(f"Environment: {os.getenv('ENVIRONMENT', 'development')}")

## 1. Load Declarative Validation Rules for Yellow Tripdata Table

In [ ]:
# Fetch validation rules for yellow_tripdata defined in config/validation_rules.yaml
rules = dq_manager.get_table_rules('yellow_tripdata')
print("Loaded validation rules for table 'yellow_tripdata':")
print(rules)

## 2. Load Ingestion Parquet File & Validate Quality

In [ ]:
# Read yellow_tripdata parquet dataset from data-files/
df = dq_manager.load_dataset('yellow_tripdata', stage_name='ingestion')
print(f"Loaded dataset shape: {df.shape}")
print(df.head())

# Convert DataFrame to Great Expectations PandasDataset
gx_df = gx.from_pandas(df)

# Dynamically apply rules configured in config/validation_rules.yaml
dq_manager.apply_table_rules(gx_df, 'yellow_tripdata')

# Evaluate Validation Results
validation_results = gx_df.validate()
passed = dq_manager.evaluate_validation_results(validation_results)

print(f"Data Quality Checks Passed: {passed}")